<a href="https://colab.research.google.com/github/Frans1828/scikit-learn-cookbook/blob/main/04.%20Models%20with%20Distance%20Metrics%20and%20Nearest%20Neighbors/ch04_knn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## Identitas Mahasiswa

| Informasi | Keterangan |
|---|---|
| **Nama** | Fransisco Sitepu |
| **NIM** | 101032300004 |
| **Kelas** | TK 47 03 |
| **Mata Kuliah** | Machine Learning |
| **Institusi** | Telkom University |

---


# Chapter 4: Models with Distance Metrics and Nearest Neighbors

## Pengantar: Instance-Based Learning

K-Nearest Neighbors (KNN) adalah algoritma **instance-based learning** (lazy learning) yang tidak membangun model eksplisit saat training. Prediksi dibuat langsung berdasarkan **kemiripan** data baru dengan data training.

**Intuisi**: untuk mengklasifikasikan titik data baru $\mathbf{x}$, cari $K$ tetangga terdekat dari training set, lalu ambil **voting mayoritas** (klasifikasi) atau **rata-rata** (regresi).

### Keputusan Klasifikasi KNN

$$\hat{y} = \text{mode}\Big(\{y_i : \mathbf{x}_i \in N_k(\mathbf{x})\}\Big)$$

di mana $N_k(\mathbf{x})$ adalah himpunan $K$ tetangga terdekat dari $\mathbf{x}$.

### Regresi KNN

$$\hat{y} = \frac{1}{K} \sum_{i \in N_k(\mathbf{x})} y_i$$


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.datasets import load_iris, load_wine, load_breast_cancer, make_classification, make_regression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score, classification_report

# Load datasets
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

wine = load_wine()
X_wine, y_wine = wine.data, wine.target

cancer = load_breast_cancer()
X_cancer, y_cancer = cancer.data, cancer.target

print("=== Datasets Loaded ===")
print(f"Iris:   X={X_iris.shape}, classes={np.unique(y_iris)}")
print(f"Wine:   X={X_wine.shape}, classes={np.unique(y_wine)}")
print(f"Cancer: X={X_cancer.shape}, classes={np.unique(y_cancer)}")


## 4.1 Metrik Jarak

Pilihan metrik jarak sangat mempengaruhi performa KNN. Beberapa metrik umum:

### Euclidean Distance ($L_2$ Norm)

$$d_{\text{Eucl}}(\mathbf{x}, \mathbf{x'}) = \sqrt{\sum_{j=1}^{p} (x_j - x'_j)^2} = \|\mathbf{x} - \mathbf{x'}\|_2$$

### Manhattan Distance ($L_1$ Norm)

$$d_{\text{Manh}}(\mathbf{x}, \mathbf{x'}) = \sum_{j=1}^{p} |x_j - x'_j| = \|\mathbf{x} - \mathbf{x'}\|_1$$

### Minkowski Distance (Generalization)

$$d_{\text{Mink}}(\mathbf{x}, \mathbf{x'}) = \left(\sum_{j=1}^{p} |x_j - x'_j|^p \right)^{1/p}$$

Ketika $p=1$: Manhattan, $p=2$: Euclidean, $p \to \infty$: Chebyshev.

### Cosine Similarity

$$\text{cos}(\theta) = \frac{\mathbf{x} \cdot \mathbf{x'}}{\|\mathbf{x}\| \cdot \|\mathbf{x'}\|}$$

Berguna untuk data teks dan data sparse.


In [ ]:
from sklearn.metrics.pairwise import euclidean_distances, manhattan_distances, cosine_similarity

# Manual distance computation
def minkowski_distance(a, b, p):
    return np.sum(np.abs(a - b)**p)**(1/p)

# Compare distances between two points
point_a = np.array([1.0, 2.0, 3.0])
point_b = np.array([4.0, 6.0, 3.0])

print("=== Distance Metrics between point_a and point_b ===")
print(f"Points: a={point_a}, b={point_b}")
print(f"Euclidean (L2):  {np.sqrt(np.sum((point_a-point_b)**2)):.4f}")
print(f"Manhattan (L1):  {np.sum(np.abs(point_a-point_b)):.4f}")
print(f"Minkowski p=3:   {minkowski_distance(point_a, point_b, 3):.4f}")
print(f"Chebyshev (inf): {np.max(np.abs(point_a-point_b)):.4f}")

cos_sim = np.dot(point_a, point_b) / (np.linalg.norm(point_a) * np.linalg.norm(point_b))
print(f"Cosine similarity: {cos_sim:.4f}")
print(f"Cosine distance:   {1 - cos_sim:.4f}")

# Visualize unit balls for different Minkowski distances
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
theta = np.linspace(0, 2*np.pi, 1000)

for ax, p, title in zip(axes, [1, 2, 3, 10], ['L1 (Manhattan)', 'L2 (Euclidean)', 'L3 (Minkowski)', 'L10 (quasi-Chebyshev)']):
    # Find points x,y such that (|x|^p + |y|^p)^(1/p) = 1
    x = np.linspace(-1.5, 1.5, 1000)
    y_pos = (np.maximum(1 - np.abs(x)**p, 0))**(1/p)
    ax.fill_between(x, -y_pos, y_pos, alpha=0.3, color='steelblue')
    ax.plot(x, y_pos, 'b-', lw=2)
    ax.plot(x, -y_pos, 'b-', lw=2)
    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
    ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
    ax.set_aspect('equal')
    ax.set_title(f'Unit Ball: {title}')

plt.tight_layout()
plt.show()


## 4.2 KNN Classifier

### Bias-Variance Tradeoff dan Nilai K

Pilihan nilai $K$ sangat menentukan performa:
- **K kecil** (misalnya $K=1$): model sangat fleksibel → **high variance, low bias** → overfitting
- **K besar** (misalnya $K=n$): model sangat smooth → **low variance, high bias** → underfitting

Nilai $K$ optimal dipilih melalui cross-validation.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Standardize data
scaler = StandardScaler()
X_iris_sc = scaler.fit_transform(X_iris)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_iris_sc, y_iris, test_size=0.3, random_state=42, stratify=y_iris
)

# Test different K values
k_range = range(1, 31)
train_scores = []
test_scores = []
cv_scores = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_tr, y_tr)
    train_scores.append(knn.score(X_tr, y_tr))
    test_scores.append(knn.score(X_te, y_te))
    cv = cross_val_score(knn, X_iris_sc, y_iris, cv=5).mean()
    cv_scores.append(cv)

# Plot bias-variance tradeoff
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_range, train_scores, 'b-o', markersize=4, label='Train accuracy', alpha=0.8)
ax.plot(k_range, test_scores, 'r-s', markersize=4, label='Test accuracy', alpha=0.8)
ax.plot(k_range, cv_scores, 'g-^', markersize=4, label='CV accuracy (5-fold)', alpha=0.8)
ax.axvline(x=k_range[np.argmax(cv_scores)], color='orange', linestyle='--',
           label=f'Best K={k_range[np.argmax(cv_scores)]} (CV)')
ax.set_xlabel('Number of Neighbors K')
ax.set_ylabel('Accuracy')
ax.set_title('KNN: Effect of K on Accuracy (Iris Dataset)')
ax.legend()
ax.set_xlim(0, 31)

# Add annotations
ax.annotate('Overfitting\n(K too small)', xy=(1, train_scores[0]),
            xytext=(5, 0.97), arrowprops=dict(arrowstyle='->', color='blue'), fontsize=9)
ax.annotate('Underfitting\n(K too large)', xy=(30, train_scores[-1]),
            xytext=(22, 0.90), arrowprops=dict(arrowstyle='->', color='red'), fontsize=9)
plt.tight_layout()
plt.show()

best_k = k_range[np.argmax(cv_scores)]
print(f"Best K (by 5-fold CV): {best_k}, CV accuracy: {max(cv_scores):.4f}")


In [ ]:
# Decision boundary visualization (use first 2 features for viz)
X_2d = X_iris_sc[:, :2]  # use only first 2 features

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors_bg = ListedColormap(['#FFAAAA', '#AAFFAA', '#AAAAFF'])
colors_pt = ListedColormap(['red', 'green', 'blue'])

for ax, k in zip(axes, [1, 7, 25]):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_2d, y_iris)

    xx, yy = np.meshgrid(
        np.linspace(X_2d[:,0].min()-0.5, X_2d[:,0].max()+0.5, 200),
        np.linspace(X_2d[:,1].min()-0.5, X_2d[:,1].max()+0.5, 200)
    )
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.4, cmap=colors_bg)
    ax.scatter(X_2d[:, 0], X_2d[:, 1], c=y_iris, cmap=colors_pt, s=20, edgecolors='gray', linewidth=0.3)
    ax.set_title(f'KNN Decision Boundary (K={k})\nCV acc={cross_val_score(knn, X_2d, y_iris, cv=5).mean():.3f}')
    ax.set_xlabel('Feature 1 (scaled)')
    ax.set_ylabel('Feature 2 (scaled)')

plt.tight_layout()
plt.show()


## 4.3 KNN Regressor

Untuk regresi, prediksi adalah **rata-rata** (atau rata-rata berbobot) nilai target dari $K$ tetangga terdekat:

$$\hat{y}(\mathbf{x}) = \frac{\sum_{i \in N_k(\mathbf{x})} w_i y_i}{\sum_{i \in N_k(\mathbf{x})} w_i}$$

- **Uniform weights** ($w_i = 1$): semua tetangga memiliki bobot sama
- **Distance weights** ($w_i = 1/d_i$): tetangga yang lebih dekat memiliki bobot lebih besar


In [ ]:
from sklearn.neighbors import KNeighborsRegressor

# Create 1D regression data
np.random.seed(42)
X_reg = np.sort(np.random.uniform(0, 2*np.pi, 80)).reshape(-1, 1)
y_reg = np.sin(X_reg.ravel()) + np.random.randn(80) * 0.2

X_plot = np.linspace(0, 2*np.pi, 300).reshape(-1, 1)
y_true = np.sin(X_plot.ravel())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
configs = [(3, 'uniform'), (3, 'distance'), (15, 'uniform')]

for ax, (k, weights) in zip(axes, configs):
    knn_reg = KNeighborsRegressor(n_neighbors=k, weights=weights)
    knn_reg.fit(X_reg, y_reg)
    y_pred_reg = knn_reg.predict(X_plot)

    rmse = np.sqrt(mean_squared_error(y_reg, knn_reg.predict(X_reg)))
    r2 = r2_score(y_reg, knn_reg.predict(X_reg))

    ax.scatter(X_reg, y_reg, s=20, alpha=0.5, color='gray', label='Data')
    ax.plot(X_plot, y_true, 'k--', lw=1.5, label='True sin(x)')
    ax.plot(X_plot, y_pred_reg, 'r-', lw=2, label=f'KNN pred')
    ax.set_title(f'KNN Regressor: K={k}, weights={weights}\nRMSE={rmse:.3f}, R²={r2:.3f}')
    ax.set_xlabel('X')
    ax.set_ylabel('y')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


## 4.4 Efficient Neighbor Search: KDTree dan BallTree

Pencarian brute-force untuk $K$ tetangga terdekat memiliki kompleksitas $O(n \cdot p)$ untuk setiap query. Struktur data khusus dapat mempercepat pencarian ini:

| Algoritma | Kompleksitas (approx) | Cocok Untuk |
|---|---|---|
| **Brute Force** | $O(n \cdot p)$ | Dataset kecil, fitur banyak |
| **KDTree** | $O(p \log n)$ | Fitur sedikit ($p < 20$), data berdistribusi merata |
| **BallTree** | $O(p \log n)$ | Dimensi tinggi, jarak non-Euclidean |


In [ ]:
from sklearn.neighbors import NearestNeighbors
import time

# Generate large random dataset
np.random.seed(42)
n_large = 5000
X_large = np.random.randn(n_large, 10)

# Query point
X_query = np.random.randn(100, 10)

algorithms = ['brute', 'kd_tree', 'ball_tree']
timing_results = {}

for algo in algorithms:
    nn = NearestNeighbors(n_neighbors=5, algorithm=algo, metric='euclidean')
    nn.fit(X_large)
    start = time.time()
    distances, indices = nn.kneighbors(X_query)
    elapsed = time.time() - start
    timing_results[algo] = elapsed
    print(f"{algo:12s}: {elapsed*1000:.2f} ms | Sample distances: {distances[0].round(3)}")

# Verify all algorithms find the same neighbors
nn_brute = NearestNeighbors(n_neighbors=5, algorithm='brute')
nn_ball = NearestNeighbors(n_neighbors=5, algorithm='ball_tree')
nn_brute.fit(X_large); nn_ball.fit(X_large)
_, idx_brute = nn_brute.kneighbors(X_query[:1])
_, idx_ball = nn_ball.kneighbors(X_query[:1])
print(f"\nBrute force neighbors: {idx_brute[0]}")
print(f"BallTree neighbors:    {idx_ball[0]}")
print(f"Results match: {np.array_equal(np.sort(idx_brute[0]), np.sort(idx_ball[0]))}")


## 4.5 Weighted KNN dan Radius Neighbors

### Distance-Weighted KNN

Tetangga yang lebih dekat mendapat bobot lebih besar:

$$w_i = \frac{1}{d(\mathbf{x}, \mathbf{x}_i)^2 + \epsilon}$$

### Radius Neighbors

Daripada mencari $K$ tetangga terdekat, `RadiusNeighborsClassifier` menggunakan **semua** titik dalam radius $r$:

$$N_r(\mathbf{x}) = \{\mathbf{x}_i : d(\mathbf{x}, \mathbf{x}_i) \leq r\}$$


In [ ]:
from sklearn.neighbors import RadiusNeighborsClassifier
from sklearn.pipeline import Pipeline

# Compare uniform vs distance weighting on Breast Cancer dataset
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X_cancer, y_cancer, test_size=0.3, random_state=42, stratify=y_cancer
)

# Evaluate KNN variants
print("=== KNN Variants on Breast Cancer Dataset ===")
print(f"{'Configuration':<40} {'Test Acc':>10} {'5-CV Acc':>10}")
print("-" * 62)

configs_knn = [
    ('KNN K=5, uniform, Euclidean', KNeighborsClassifier(5, weights='uniform', metric='euclidean')),
    ('KNN K=5, distance, Euclidean', KNeighborsClassifier(5, weights='distance', metric='euclidean')),
    ('KNN K=5, uniform, Manhattan', KNeighborsClassifier(5, weights='uniform', metric='manhattan')),
    ('KNN K=11, distance, Euclidean', KNeighborsClassifier(11, weights='distance', metric='euclidean')),
    ('KNN K=11, distance, Minkowski p=3', KNeighborsClassifier(11, weights='distance', p=3)),
]

for name, clf in configs_knn:
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', clf)
    ])
    pipe.fit(X_tr_c, y_tr_c)
    test_acc = accuracy_score(y_te_c, pipe.predict(X_te_c))
    cv_acc = cross_val_score(
        Pipeline([('scaler', StandardScaler()), ('knn', clf)]),
        X_cancer, y_cancer, cv=5
    ).mean()
    print(f"{name:<40} {test_acc:>10.4f} {cv_acc:>10.4f}")


In [ ]:
# Full evaluation with best KNN config
best_knn_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=11, weights='distance', metric='euclidean'))
])
best_knn_pipe.fit(X_tr_c, y_tr_c)
y_pred_c = best_knn_pipe.predict(X_te_c)

print("=== Best KNN Configuration: Classification Report ===")
print(classification_report(y_te_c, y_pred_c, target_names=cancer.target_names))

# Confusion matrix
from sklearn.metrics import ConfusionMatrixDisplay
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_estimator(
    best_knn_pipe, X_te_c, y_te_c,
    display_labels=cancer.target_names,
    cmap='Blues', ax=ax
)
ax.set_title('KNN (K=11, distance-weighted) - Confusion Matrix\nBreast Cancer Dataset')
plt.tight_layout()
plt.show()


## Ringkasan: Key Takeaways Chapter 4

| Konsep | Poin Kunci |
|---|---|
| **Instance-Based Learning** | Tidak ada model eksplisit; prediksi berdasarkan kemiripan dengan training data |
| **Metrik Jarak** | Pilihan metrik (Euclidean, Manhattan, Cosine) sangat mempengaruhi hasil KNN |
| **Nilai K** | K kecil = high variance (overfitting); K besar = high bias (underfitting) |
| **Normalisasi** | WAJIB menormalisasi fitur sebelum KNN; skala fitur yang berbeda akan mendominasi |
| **KDTree / BallTree** | Mempercepat pencarian tetangga dari O(n) ke O(log n) untuk data besar |
| **Distance Weighting** | Tetangga yang lebih dekat mendapat bobot lebih besar; umumnya meningkatkan performa |
| **Predict Proba** | `predict_proba()` memberikan proporsi kelas dari K tetangga terdekat |

### Kapan Menggunakan KNN?
- Dataset relatif kecil (<50K sampel)
- Tidak ada asumsi distribusi data
- Butuh model yang mudah diinterpretasikan secara intuitif
- Sebagai baseline sebelum mencoba model yang lebih kompleks

### Keterbatasan KNN
- **Prediction time** lambat: $O(n \cdot p)$ per query (brute-force)
- **Memory**: seluruh training set harus tersimpan
- **Curse of dimensionality**: performa menurun drastis pada dimensi sangat tinggi
- **Imbalanced data**: rentan terhadap kelas mayoritas
